In [ ]:
!pip install spacy scikit-learn nltk pandas matplotlib seaborn wordcloud plotly langdetect deep-translator sumy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys

project_path = '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM'

sys.path.append(project_path)

In [ ]:
import pandas as pd

df = pd.read_csv(f'{project_path}/data/bbc_news.csv')
df = df[:300]
df.head()

,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


In [ ]:
df = df.drop(columns=['pubDate', 'guid', 'link'])
df.head()

,title,description
0,Ukraine: Angry Zelensky vows to punish Russian...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,Consumers are feeling the impact of higher ene...


In [ ]:
from newsbot.preprocessing import clean_text, preprocess_text
from newsbot.extract_syntactic_features import extract_syntactic_features
from newsbot.pos import analyze_pos_patterns
from newsbot.sentiment_analysis import analyze_sentiment
from newsbot.ner import named_entity_recognition
from newsbot.translation import translate
from newsbot.language_detection import language_detection
from newsbot.summarize import summarize
from newsbot.intent import classify_intent, process_query, generate_response

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
import joblib

label_encoder = joblib.load(f'{project_path}/newsbot/label_encoder.pkl')

tfidf_vectorizer = joblib.load(f'{project_path}/newsbot/tfidf_vectorizer.pkl')

model = joblib.load(f'{project_path}/newsbot/svm_model.pkl')

feature_columns = joblib.load(f'{project_path}/newsbot/feature_columns.pkl')

In [ ]:
def predict_category_pipeline(text):
    cleaned_text = clean_text(text)

    preprocessed_text_from_module = preprocess_text(cleaned_text)

    tfidf_matrix = tfidf_vectorizer.transform([preprocessed_text_from_module])
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

    pos_features = analyze_pos_patterns(cleaned_text)
    pos_df = pd.DataFrame([pos_features])

    combined_df = pd.concat([tfidf_df, pos_df], axis=1)
    final_text = combined_df.reindex(columns=feature_columns, fill_value=0)

    prediction = model.predict(final_text)
    category = label_encoder.inverse_transform(prediction)[0]

    return category

In [ ]:
df['Predicted_Category'] = df['description'].apply(lambda t: predict_category_pipeline(t))

In [ ]:
df.head()

,title,description,Predicted_Category
0,Ukraine: Angry Zelensky vows to punish Russian...,The Ukrainian president says the country will ...,business
1,War in Ukraine: Taking cover in a town under a...,"Jeremy Bowen was on the frontline in Irpin, as...",business
2,Ukraine war 'catastrophic for global food',One of the world's biggest fertiliser firms sa...,business
3,Manchester Arena bombing: Saffie Roussos's par...,The parents of the Manchester Arena bombing's ...,politics
4,Ukraine conflict: Oil price soars to highest l...,Consumers are feeling the impact of higher ene...,business


In [ ]:

current_article = None

def NewsBotAgent(df):
  print('Welcome to the Newsbot. Type Goodbye to end session.')

  while True:
    query = input('\nEnter your query: ')

    if query.lower() == 'goodbye':
      print('Thank you for using the Newsbot. Goodbye!')
      break

    intent = classify_intent(query)

    result = process_query(df, intent)

    response = generate_response(result, intent)

    print(f'\nNewsbot: {response}')

In [14]:
NewsBotAgent(df)

Welcome to the Newsbot. Type Goodbye to end session.

Enter your query: show me a sport article

Newsbot: Category: sport
Sentiment: {'neg': 0.288, 'neu': 0.577, 'pos': 0.135, 'compound': -0.4939, 'sentiment_label': 'negative'}
POS tags: {'NNP': 0.21052631578947367, 'NNS': 0.05263157894736842, 'VBP': 0.05263157894736842, 'DT': 0.05263157894736842, 'NN': 0.21052631578947367, 'POS': 0.05263157894736842, 'IN': 0.10526315789473684, 'PRP$': 0.05263157894736842, 'CD': 0.10526315789473684, 'SYM': 0.05263157894736842, '.': 0.05263157894736842}
Text: Manchester United legends savage the club's performance in their 4-1 derby defeat by Manchester City....

Enter your query: translate the article to japanese

Newsbot: Translation: マンチェスター・ユナイテッドのレジェンドが、マンチェスター・シティとのダービーで4-1で敗れたクラブのパフォーマンスを酷評した。...

Enter your query: perform named entity recognition

Newsbot: Entities found: Manchester United (PERSON), 4 (CARDINAL), Manchester City (GPE)

Enter your query: show me an article of business

Newsbot: C